In [10]:
import pandas as pd
import pickle

In [2]:
df = pd.read_csv("Source data/train_data.csv")
df.head(5)

,year,month,day,order,country,session_id,page1_main_category,page2_clothing_model,colour,location,model_photography,price,price_2,page
0,2008,6,22,21,29,15648,3,C20,13,1,2,48,1,2
1,2008,5,19,6,29,10018,2,B26,13,3,1,57,1,2
2,2008,7,15,2,29,19388,3,C13,9,5,1,48,1,1
3,2008,5,2,2,29,7181,2,B11,2,4,1,43,2,1
4,2008,6,9,16,29,13493,2,B31,9,5,1,57,1,2


Feature Engineering and Data preprocessing

In [ ]:
# Define the processing function
def preprocess_pipeline(df):
    # Total clicks per session
    total_clicks = df.groupby('session_id')['order'].count().reset_index()
    total_clicks.rename(columns={'order': 'total_clicks'}, inplace=True)

    # Browsing depth per session
    browsing_depth = df.groupby('session_id')['page'].max().reset_index()
    browsing_depth.rename(columns={'page': 'browsing_depth'}, inplace=True)

    # Merge metrics into original DataFrame
    session_metrics = total_clicks.merge(browsing_depth, on='session_id')
    df = df.merge(session_metrics, on='session_id')

    # Create weekday and weekend flags
    df['weekday'] = pd.to_datetime(df[['year', 'month', 'day']]).dt.dayofweek
    df['weekend'] = (df['weekday'] >= 5).astype(int)

    # Drop unnecessary columns
    df.drop(columns=['year', 'session_id', 'page2_clothing_model'], inplace=True)

    return df

# Pickle the function
with open('/Users/somesh-19583/Desktop/Customer conversion/preprocessing_pipeline.pkl', 'wb') as f:
    pickle.dump(preprocess_pipeline, f)

df = pd.read_csv("Source data/train_data.csv")

# Load the pipeline back (for demonstration)
with open('/Users/somesh-19583/Desktop/Customer conversion/preprocessing_pipeline.pkl', 'rb') as f:
    pipeline = pickle.load(f)

# Apply the pipeline
processed_df = pipeline(df)

processed_df

,month,day,order,country,page1_main_category,colour,location,model_photography,price,price_2,page,total_clicks,browsing_depth,weekday,weekend
0,6,22,21,29,3,13,1,2,48,1,2,84,4,6,1
1,5,19,6,29,2,13,3,1,57,1,2,9,2,0,0
2,7,15,2,29,3,9,5,1,48,1,1,10,3,1,0
3,5,2,2,29,2,2,4,1,43,2,1,6,2,4,0
4,6,9,16,29,2,9,5,1,57,1,2,15,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132374,7,4,3,29,4,2,1,1,48,1,2,5,5,4,0
132375,6,19,9,29,3,14,3,1,28,2,2,33,5,3,0
132376,7,15,4,29,1,3,2,2,38,2,1,8,1,1,0
132377,7,28,16,29,3,9,5,2,20,2,3,18,4,0,0
